# PayPal - One Touch Conversion Performance for Mobile Shoppers

In [1]:
import pandas as pd
import numpy as np
import polars as pl
from datetime import date

In [2]:
df_mobile = pd.read_csv('../Data/017/fct_mobile_transactions.csv', parse_dates=['transaction_date'])

pl_mobile = pl.read_csv('../Data/017/fct_mobile_transactions.csv', try_parse_dates=True)


# Pregunta 1

### Para nuestro análisis de la función PayPal One Touch, ¿cuál es el número total de transacciones móviles que utilizaron 'One Touch' durante julio de 2024? Notarás que el campo login_method no tiene una capitalización consistente (mayúsculas/minúsculas), ¡así que asegúrate de tener esto en cuenta en tu consulta!

```SQL
SELECT
    COUNT(*)
FROM fct_mobile_transactions
WHERE ((EXTRACT(MONTH FROM transaction_date) = 7) AND
       (EXTRACT(YEAR FROM transaction_date) = 2024)) AND
    LOWER(TRIM(login_method)) = 'one touch';
```

In [3]:
res = df_mobile[
    (df_mobile['transaction_date'].dt.month == 7) &
    (df_mobile['transaction_date'].dt.year == 2024) &
    (df_mobile['login_method'].str.strip().str.lower() == 'one touch')
].shape[0]

res

9

In [8]:
res = pl_mobile.filter(
    (pl.col('transaction_date').dt.month() == 7) &
    (pl.col('transaction_date').dt.year() == 2024) &
    (pl.col('login_method').str.strip_chars().str.to_lowercase() == 'one touch')
).height

res

9

# Pregunta 2

### Para determinar la adopción de la función One Touch por parte de los usuarios, ¿cuántos usuarios distintos tuvieron transacciones móviles exitosas utilizando One Touch durante julio de 2024? Renombra la columna del conteo de usuarios como 'Unique_Users'. Esta información respaldará nuestra investigación sobre el compromiso (engagement) con las transacciones.

```SQL
SELECT
    COUNT(DISTINCT user_id) AS Unique_Users
FROM fct_mobile_transactions
WHERE ((EXTRACT(MONTH FROM transaction_date) = 7) AND
       (EXTRACT(YEAR FROM transaction_date) = 2024)) AND
    LOWER(TRIM(login_method)) = 'one touch' AND
    transaction_status = 'Success';
```

In [13]:
res = df_mobile[
    (df_mobile['transaction_date'].dt.month == 7) &
    (df_mobile['transaction_date'].dt.year == 2024) &
    (df_mobile['login_method'].str.strip().str.lower() == 'one touch') &
    (df_mobile['transaction_status'] == 'Success')
]['user_id'].nunique()

res

7

In [21]:
res = pl_mobile.filter(
    (pl.col('transaction_date').dt.month() == 7) &
    (pl.col('transaction_date').dt.year() == 2024) &
    (pl.col('login_method').str.strip_chars().str.to_lowercase() == 'one touch') &
    (pl.col('transaction_status') == 'Success')
).select(
    Unique_Users = pl.col('user_id').n_unique()
)

res

Unique_Users
u32
7


# Pregutna 3

### Queremos entender la adopción de las funciones One Touch frente a Standard. ¿Cuántas transacciones exitosas hubo en julio de 2024 para One Touch y Standard respectivamente? Recuerda que los datos en login_method tienen un uso de mayúsculas inconsistente, ¡así que queremos solucionar esto!

```SQL
SELECT
    COUNT(CASE WHEN LOWER(TRIM(login_method)) = 'one touch' THEN 1 END) AS one_touch_success,
    COUNT(CASE WHEN LOWER(TRIM(login_method)) = 'standard' THEN 1 END) AS standard_success
FROM fct_mobile_transactions
WHERE ((EXTRACT(MONTH FROM transaction_date) = 7) AND
       (EXTRACT(YEAR FROM transaction_date) = 2024)) AND
    LOWER(TRIM(login_method)) in ('one touch', 'standard') AND
    transaction_status = 'Success';
```

In [25]:
res = df_mobile[
    (df_mobile['transaction_date'].dt.month == 7) &
    (df_mobile['transaction_date'].dt.year == 2024) &
    (df_mobile['transaction_status'] == 'Success')
].copy()

# Primero normalizamos la columna para poder comparar
res['method_clean'] = res['login_method'].str.strip().str.lower()

# Replicamos las columnas de SQL
final_res = pd.DataFrame({
    'one_touch_success': [(res['method_clean'] == 'one touch').sum()],
    'standard_success': [(res['method_clean'] == 'standard').sum()]
})

print(final_res)


   one_touch_success  standard_success
0                  8                 4


In [28]:
res_polars = (
    pl_mobile
    .filter(
        (pl.col('transaction_date').dt.month() == 7) &
        (pl.col('transaction_date').dt.year() == 2024) &
        (pl.col('transaction_status') == 'Success')
    )
    .with_columns(
        method = pl.col('login_method').str.strip_chars().str.to_lowercase() 
    )
    .filter(pl.col('method').is_in(['one touch', 'standard']))
    .group_by('method')
    .len()
)

res_polars

method,len
str,u32
"""one touch""",8
"""standard""",4
